# Munich rents: explaining a fit with a language model

A fitted boosting model is an offset (the constant from which boosting starts), the learners the path selected and one
coefficient vector per learner. `llm_explain_prompt()` serializes those numbers
into text a language model can read, and `llm_describe()` sends it and returns
the reply. This case runs that pipeline on the 1999 Munich rent survey. The reply summarizes the fitted effects, suggests explanations and proposes
further checks. We compare those explanations with the model output and data.

A single prompt pasted into a chat box tends to come back as a walk through
each effect, because the prompt's bulk is tables and a reply mirrors the
structure of what it is given. `llm_describe(reasoning=True)` therefore holds
one three-turn conversation: read the effects, hunt for what does not fit,
then compress. The page shows the reply that conversation produced, pass by
pass, and then reads it the way a statistician should: what holds up, and what
is fluency.

The reply is a stored output of one call to `llm_describe(reasoning=True)`, saved in `llm_explains_a_model.json`, because a live call needs an API key and never returns the same text twice.


:::{admonition} What this study estimates
:class: dropdown

| | |
|---|---|
| **Target** | an analyst's brief a language model writes from a fitted model |
| **Observed response** | `rentsqm`, net rent per square metre |
| **Estimand** | none is estimated here — the study is about rendering an existing fit, not about a new quantity |
| **Units** | euros per square metre per month |
| **Sampling unit** | one apartment |
| **Dependence** | clustered by district |
| **Split / resampling unit** | not applicable; nothing is resampled |
| **Tuning** | the fit is taken as given |
| **Uncertainty** | not quantified; the deliverable is a description, and the study warns that a fluent description is not evidence |
| **Claim type** | descriptive of the fitted model itself |
:::


## The fit

`munich_rent` holds the 1999 Munich rent survey: 3082 apartments in 336
districts, with floor area, construction year, an above-standard bathroom and
kitchen indicator, a central-heating indicator (named `cheating`, its
`gamlss.data` name), a location grade and the district. German tenancy law
allows a landlord to raise rent toward the local reference level — the
Mietspiegel — and no further, so the fitted value for an apartment is the
figure both parties refer to.

The formula gives each predictor a learner suited to its type: penalised
B-splines for area and construction year, linear terms for the indicators and
the location grade, and a Markov random field for the district (one effect per district, each shrunk toward the effects of its neighbours), which smooths
district effects toward neighbouring districts through the bundled adjacency
graph.


In [1]:
import numpy as np
import polars as pl

import mboostrs as mb

rent = mb.datasets.munich_rent()
print(f"{rent.height} apartments in {rent['district'].n_unique()} districts")
print(f"rent per m2: mean {rent['rentsqm'].mean():.2f}, "
      f"sd {rent['rentsqm'].std():.2f}, "
      f"range [{rent['rentsqm'].min():.2f}, {rent['rentsqm'].max():.2f}]")


3082 apartments in 336 districts
rent per m2: mean 7.11, sd 2.44, range [0.42, 17.72]


In [2]:
adjacency = mb.datasets.munich_rent_adjacency()

# Integer-code districts to 0..335 for bmrf.
codes = sorted(rent["district"].unique().to_list())
position = {d: i for i, d in enumerate(codes)}
rent = rent.with_columns(
    pl.col("district").replace_strict(position).alias("dcode")
)

# Build the graph Laplacian from the adjacency list.
n_d = len(codes)
K = np.zeros((n_d, n_d))
for a, b in zip(adjacency["a"], adjacency["b"]):
    i, j = position[a], position[b]
    K[i, j] = K[j, i] = -1.0
    K[i, i] += 1.0
    K[j, j] += 1.0

formula = (
    "rentsqm ~ bbs(area, df=4) + bbs(yearc, df=4) + bols(location) + "
    "bols(bath) + bols(kitchen) + bols(cheating) + bmrf(dcode, K=K_mat)"
)
fit = mb.gamboost(
    formula, data=rent, family=mb.Gaussian(),
    control=mb.boost_control(mstop=2000, nu=0.1),
    env={"K_mat": K},
)
print(f"mstop={fit.mstop}, risk={fit.risk_[-1]:.2f}")


mstop=2000, risk=10802.64


Construction year and floor area carry 72% of the risk reduction between
them. The pattern to watch is the district field: it takes 78% of the updates
and 6.2% of the credit, and the reply below treats that gap as the main pattern to explain.


In [3]:
fi = fit.varimp()
for name, imp in sorted(fi["reduction"].items(), key=lambda x: -x[1]):
    print(f"  {name:20s} {imp:.3f}")


  yearc                0.376
  area                 0.340
  cheating             0.147
  dcode                0.062
  location             0.054
  kitchen              0.015
  bath                 0.006


## The prompt

The prompt is a serialization of the fit, written in six sections: `reading`
(how an additive boosting fit is read), `facts` (size, offset, response
scale, the data description the caller supplies), `importance` (the ranked
learners), `effects` (one block per selected learner), `caveats` (what the
fit cannot say) and `task` (the instruction the reply follows). Every number
comes from methods the model already exposes — `summary`, `varimp`,
`partial_effects`, `coef` — and nothing is sent anywhere until the caller
sends it. The full text runs about ten thousand characters; the cell shows
each section's size and first line, and one effect block in full.


In [4]:
dataset_description = (
    "Munich rent survey 1999: 3082 rent-controlled apartments. "
    "Response is net rent per square metre (EUR). Predictors: "
    "floor area, construction year, location (1 average, 2 good, "
    "3 top; 78 apartments are top), indicators of an above-standard "
    "bathroom and kitchen, a central-heating indicator (named "
    "'cheating' in the data, its gamlss.data name), "
    "and one of 336 city districts."
)
response_description = "net rent per square metre in EUR"

for section in ("reading", "facts", "importance", "effects", "caveats", "task"):
    text = fit.llm_explain_prompt(
        include=(section,), dataset_description=dataset_description,
        response_description=response_description)
    first = text.splitlines()[0][:60]
    print(f"{section:<11} {len(text):>6,} characters   {first}")

# The full prompt is built once; the reply section checks its hash.
prompt = fit.llm_explain_prompt(dataset_description=dataset_description,
                                response_description=response_description)


reading      1,397 characters   How to read what follows.
facts        2,647 characters                             Gaussian boosting results
importance     486 characters   Base learners, ranked:
effects      3,135 characters   Effects (7 of 7 selected learners, most important first):
caveats      1,582 characters   Limits of this description.
task         1,241 characters   Write the brief a statistician would hand a colleague who ha


One effect block, construction year. It names the learner and its basis,
how often the path selected it and its share of the risk reduction, the
covariate's observed range, the contribution on a grid of covariate values,
and the shape in one line — everything a reader needs to reconstruct the
effect without seeing the plot.


In [5]:
print(fit.llm_explain_prompt(include=("effects",), which=["yearc"], n_points=8))


Effects (1 of 7 selected learners, most important first):

Learner: yearc  [bbs: B-spline basis of degree 3 on 20 inner knots, df=4, second-difference penalty]
Selected in 73 of 2000 updates (3.6%), first at iteration 1
Importance: 37.6% of total risk reduction
Covariate: yearc (observed range [1918, 1997])
Contribution: {"1918": -0.573, "1929": -0.709, "1941": -0.81, "1952": -0.705, "1963": -0.217, "1974": 0.64, "1986": 1.603, "1997": 2.368}
Contribution range: [-0.81, 2.368] (span 3.178)
Shape: 1 turning point; highest near 1997, lowest near 1941


## Three turns

A model given the prompt alone tends to mirror its structure: tables in, a
walk through each table out. The `task` section works against that — it asks
for a four-part brief (findings, surprising patterns, what could mislead,
next check) and says a reply that rebuilds the tables fails — but one turn
still has to do two jobs at once: find what matters and write it down.
`llm_describe(reasoning=True)` splits the work across three turns of one
conversation:

1. the prompt, ending with the task;
2. a hunt for what a first reading misses — surprising regions, artefacts,
   misleading importance shares — with a judgment and a settling check for
   each;
3. a compression to six sentences for a reader who has the charts.

The two instructions below are printed verbatim, so the reply on this page
can be read against what was actually asked.


In [6]:
import textwrap

plan = fit.llm_explain_prompt(output="turns", num_sentences=6)
for label, text in (("Turn 2, the hunt", plan["turns"][1]),
                    ("Turn 3, the compression", plan["turns"][2])):
    print(f"{label}:")
    print(textwrap.fill(text, 78, initial_indent="  ", subsequent_indent="  "))
    print()


Turn 2, the hunt:
  Now examine the patterns you reported and the fit behind them, and surface
  what a first reading misses: regions surprising or counterintuitive for this
  subject matter, effects that look like data artefacts rather than
  mechanisms, and importance shares or update counts that could mislead —
  credit split between correlated covariates, many updates buying little risk
  reduction, a learner absorbing the signal of a variable absent from the
  model. For each item say which numbers make it notable, which explanations
  could produce it — a real mechanism, a data artefact such as a sparse
  covariate range, a covariate standing in for another, confounding by
  something not in the model, or the path having stopped where it did — which
  one you consider most likely on subject-matter grounds, and what
  observation, analysis or experiment would settle it. The fit alone cannot
  settle it, so commit to a judgment instead of listing possibilities.

Turn 3, the compres

## The reply

The cell prints the endpoint, model and date of the recorded conversation and
whether the stored prompt is the one the current package builds; the brief
that follows is the third turn's reply, verbatim.


In [7]:
import hashlib
import json
from pathlib import Path

import mistune
from IPython.display import HTML, display

artifact = json.loads(Path("llm_explains_a_model.json").read_text())
replies = [m["content"] for m in artifact["conversation"] if m["role"] == "assistant"]
local_hash = hashlib.sha256(prompt.encode()).hexdigest()[:16]
status = ("the current package builds the same prompt"
          if local_hash == artifact["prompt_sha256"]
          else "the prompt has changed since this conversation was recorded")
print(f"{artifact['llm']} at {artifact['base_url']}, {artifact['requested_at']}")
print(f"stored prompt {artifact['prompt_sha256']}, current {local_hash}: {status}")

# MyST drops text/markdown outputs; HTML renders in both the HTML and PDF builds.
render = mistune.create_markdown(escape=False, plugins=["table"])
display(HTML(render(replies[2])))


nvidia/nemotron-3-ultra-550b-a55b at https://integrate.api.nvidia.com/v1, 2026-10-04T18:48:37+00:00
stored prompt bccecc421441f3d8, current 3abbe98587f79da0: the prompt has changed since this conversation was recorded


## Reading the reply

Every number in the brief traces to a table on this page: the 78% of updates
against 6.2% of importance, the 1.81 EUR/m² heating gap, the entry iterations.
The judgments on top of them are the model's own, and they are where a reader
should slow down. Three habits are worth separating.

- **A real pattern with an invented mechanism.** The upturn of the area effect
  above 130 m² is in the fit, and the sparsity explanation is right: 45 of
  3082 apartments are larger than 130 m², one line to check. But one of the reply's judgments explains the bend with "the final segment is pulled by the penalty
  anchor", which is not how a second-difference penalty behaves. The observed count supports checking the sparse tail; it does not support
  the reply's explanation of the penalty.
- **Subject-matter claims asserted as fact.** "Central heating rare pre-1960,
  near-universal post-1980" reads like knowledge of this survey, but the model
  brought it from the world, not the prompt. It happens to hold — 20% of
  pre-1960 apartments lack central heating, 0% of post-1980 ones — and the
  check is one `group_by`. A claim in the same voice that did not hold would
  read exactly the same.
- **Judgments are testable, not tested.** That the district field's four
  degrees of freedom oversmooth 336 districts, and that the heating indicator
  proxies renovation rather than acting alone, are exactly the claims its own
  proposed checks would settle: raise the field's df and watch its importance,
  fit construction year alone and then add heating. Until those run, they are
  well-formed hypotheses.

The data description must name each variable correctly, because the language model reads it as given. For example, a description that calls the central-heating indicator (`cheating`, its name in `gamlss.data`) a "rent-cheating indicator" leads the reply to explain a 1.81 EUR/m² gap as rent fraud and to propose instrumental variables for it. A wrong label produces a confident wrong explanation.


## The working passes

The brief was compressed from two longer replies, shown here so the
compression can be audited: the first carries the four-part task, the second
the hunt with its judgments and settling checks.


In [8]:
display(HTML("<h3>First reply: findings</h3>" + render(replies[0])))


In [9]:
display(HTML("<h3>Second reply: the hunt</h3>" + render(replies[1])))


## Calling `llm_describe`

`llm_describe()` posts to an OpenAI-compatible chat-completions endpoint.
`base_url` names the endpoint (default `https://integrate.api.nvidia.com/v1`;
vLLM, Ollama and OpenAI servers work the same way), `api_key` the bearer token
(default: the first of `MBOOSTRS_LLM_API_KEY`, `NVIDIA_API_KEY` and
`OPENAI_API_KEY` that is set) and `llm` the model id (default
`nvidia/nemotron-3-ultra-550b-a55b`). The call below produced the recorded
conversation; it is not executed here.


In [10]:
# conversation = fit.llm_describe(
#     dataset_description=dataset_description,
#     response_description=response_description,
#     reasoning=True, num_sentences=6,
#     timeout=900, max_tokens=16000,
#     return_conversation=True,
# )


## Prompt verification and interpretation limits

- The prompt is a faithful rendering of the fit: every number in it comes from
  the fitted path.
- The reply is one model's reading of those numbers. A fluent explanation of a
  pattern is not evidence that the pattern is real; the fit carries no standard
  errors, and the selected learners were estimated on the data that selected
  them.
- Importance is a share of risk reduction, not causal attribution. With
  correlated covariates the learner that reduced the risk most need not be the
  one that drives the response.
- The useful output is the list of checks: the reply's suggestions are
  hypotheses to test with the package (cross-validated `mstop`, stability
  selection, refits on subsets), not conclusions.


## R interface

The R frontend has the same two functions, and the fixed sections are
worded identically, so a prompt built in either language reads the same to
the model receiving it. `llm_describe()` posts through the **curl** package
when it is installed and the `curl` command-line tool otherwise.

```r
fit <- gamboost(
    rentsqm ~ bbs(area, df = 4) + bbs(yearc, df = 4) + bols(location) +
    bols(bath) + bols(kitchen) + bols(cheating) + bmrf(dcode, K = K_mat),
    data = rent, family = Gaussian(), control = boost_control(mstop = 2000, nu = 0.1),
    weights = NULL, offset = NULL, wargs = list(K = K)
)
cat(llm_explain_prompt(fit))
llm_describe(fit, reasoning = TRUE)
```
